In [ ]:
from utils import *
import array 

from pydub import AudioSegment
import tensorflow as tf

from keras.models import Model, Sequential
from keras.layers import Input, Conv2D, Flatten, GlobalMaxPooling2D , GlobalMaxPooling1D, MaxPooling2D, Activation, BatchNormalization, GlobalAveragePooling2D, GlobalMaxPool2D, concatenate, Dense, Dropout
from keras.optimizers import Adam
from tensorflow.python.keras.utils import to_categorical

from keras.layers import Input, GRU, RepeatVector, BatchNormalization, TimeDistributed, Conv1D
from keras.layers import GlobalAveragePooling1D, LSTM, MaxPooling1D
from keras import backend as K
from keras.layers import  Conv2D, MaxPooling2D, UpSampling2D, Lambda, Reshape
import keras
from keras.layers import AveragePooling1D
from keras.layers import ConvLSTM2D, Bidirectional, CuDNNLSTM, LSTM

from keras_tqdm import TQDMNotebookCallback
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numba

import dask.dataframe as dd
from dask.multiprocessing import get

%matplotlib inline

In [ ]:
POSSIBLE_LABELS = 'yes no up down left right on off stop go silence unknown'.split()
id2name = {i: name for i, name in enumerate(POSSIBLE_LABELS)}
name2id = {name: i for i, name in id2name.items()}
len(id2name)

In [ ]:
%%time 

train_df = pickle.load( open("cache/train_df_256_aug.pik","rb"))
valid_df = pickle.load( open("cache/valid_df_256.pik","rb"))
silent_df = pickle.load(open("cache/silent_df_256.pik","rb"))
unknown_df = pickle.load(open("cache/unknown_df_256_aug.pik","rb"))
test_df =  pickle.load(open("cache/test_df_256.pik","rb"))

In [ ]:
train_df.reset_index(inplace=True)
valid_df.reset_index(inplace=True)
unknown_df.reset_index(inplace=True)
silent_df.reset_index(inplace=True)

In [ ]:
test_preds = np.load("cache/predictions_conv2dlstm_plus_longblend1.npy")

In [ ]:
#no augmentation since the auto encoder has already seen all the train AND test files 

def train_generator(train_batch_size):
    while True:
        
        this_train = train_df.groupby('label_id').apply(lambda x: x.sample(n = 2000))
        extra_data_size = int(this_train.shape[0]* 0.1)
        this_train = pd.concat([silent_df.sample(extra_data_size),
                                this_train,
                                unknown_df.sample(extra_data_size*2)],axis=0 )
        
        this_train.reset_index(drop=True,inplace=True)
        
        shuffled_ids = random.sample(range(this_train.shape[0]), this_train.shape[0])
        
        for start in range(0, len(shuffled_ids), train_batch_size):
            x_batch = []
            y_batch = []
            
            end = min(start + train_batch_size, len(shuffled_ids))
            i_train_batch = shuffled_ids[start:end]
            for i in i_train_batch:
                x_batch.append(this_train.loc[i,'raw'].T)
#                 x_batch.append(process_wav_file(this_train.iloc[i], augment=True).T)

                y_batch.append(this_train.label_id.values[i])
                
            x_batch = 1.- np.array(x_batch)/-80.
            y_batch = to_categorical(y_batch, num_classes = len(POSSIBLE_LABELS))
            
            yield x_batch, y_batch

In [ ]:
%time t = next(train_generator(256))[0][0].T

In [ ]:
def valid_generator(val_batch_size):
    while True:
        ids = list(range(valid_df.shape[0]))
        for start in range(0, len(ids), val_batch_size):
            x_batch = []
            y_batch = []
            end = min(start + val_batch_size, len(ids))
            i_val_batch = ids[start:end]
            for i in i_val_batch:
                x_batch.append(valid_df.loc[i,'raw'].T)
                y_batch.append(valid_df.label_id.values[i])

            x_batch = 1.- np.array(x_batch)/-80.
            y_batch = to_categorical(y_batch, num_classes = len(POSSIBLE_LABELS))
            yield x_batch, y_batch

### forcing test predictions (hard - not soft)

In [ ]:
to_categorical(name2id["unknown"],num_classes = len(POSSIBLE_LABELS))[0]

In [ ]:
test_preds[np.where(test_preds.max(axis=1)<0.5)[0]] = to_categorical(name2id["unknown"],num_classes = len(POSSIBLE_LABELS))[0]

test_preds = to_categorical(np.argmax(test_preds, axis=1),num_classes = len(POSSIBLE_LABELS))


In [ ]:
# test_df["id"] = test_df.index.values

In [ ]:
# test_df.reset_index(drop=True,inplace=True)

# test_df.drop(["level_0","index"],axis=1,inplace=True)

In [ ]:
def test_pseudo_generator(test_batch_size):
    this_test = test_df #.sample(int(train_df.shape[0]//5* 0.1))
    this_test["id"] = this_test.index.values
    
#     this_test.reset_index(inplace=True)
    while True:

        shuffled_ids = random.sample(range(test_df.shape[0]), test_df.shape[0])

        for start in range(0, len(test_df), test_batch_size):
            x_batch = []
            y_batch = []

            end = min(start + test_batch_size, len(shuffled_ids))
            i_test_batch = shuffled_ids[start:end]

            for i in i_test_batch:
                x_batch.append(test_df.loc[i,'raw'].T)
    #                 x_batch.append(process_wav_file(this_train.iloc[i], augment=True).T)

                y_batch.append(test_preds[test_df.loc[i,'id']])

            x_batch = 1.- np.array(x_batch)/-80.
            y_batch = np.array(y_batch)

            yield x_batch, y_batch

In [ ]:
next(test_pseudo_generator(14))[1].max()

In [ ]:
class MixIterator(object):
    def __init__(self, iters):
        self.iters = iters
        self.multi = type(iters) is list

        self.N = 64 

    def reset(self):
        for it in self.iters: it.reset()

    def __iter__(self):
        return self

    def next(self, *args, **kwargs):

        nexts = [next(it) for it in self.iters]
#         print nexts[0][0].shape, nexts[1][0].shape
        n0 = np.concatenate([n[0] for n in nexts])
        n1 = np.concatenate([n[1] for n in nexts])
        return (n0, n1)

In [ ]:
mi = MixIterator([train_generator(50), test_pseudo_generator(14)])


In [ ]:
# next(mi)

In [ ]:
valid_df.label_id.value_counts()

In [ ]:
%time t = next(valid_generator(64))

In [ ]:
valid_df.loc[0,'raw'].shape

In [ ]:
t[0].shape

In [ ]:
#a common practice is to choose a filter size in time which spans 2/3 o

In [ ]:
p = 0.3


### Model

In [ ]:
def batch_relu(x):
    x = BatchNormalization()(x)    
    x = Activation('relu')(x)
    
    return x 

In [ ]:
p = 0.

def get_conv_stacks( x_in, filter_size=2):
    
#     x = BatchNormalization()(x_in)
    
    x =  Reshape((timesteps, input_dim,1))(x_in)
    
    
    
    x = Conv2D(32,(7,7),padding='same')(x) #was 32
    x = batch_relu(x)

    x = Conv2D(64,(3,3),padding='same')(x)
    x = batch_relu(x)

    
    x = MaxPooling2D((1,3))(x)
    
    x = Conv2D(64,(3,3),padding='same')(x)
    x = batch_relu(x)
    x = Conv2D(64,(3,3),padding='same')(x)
    x = batch_relu(x)

    
    
    x = MaxPooling2D((2,3))(x)
    

    x = Conv2D(128,(3,3),padding='same')(x)
    x = batch_relu(x)
    x = Conv2D(128,(3,3),padding='same')(x)
    x = batch_relu(x)

    x = MaxPooling2D((1,3))(x)
    
    
    x = Conv2D(128,(3,3),padding='same')(x)
    x = batch_relu(x)
    x = Conv2D(128,(3,3),padding='same')(x)
    x = batch_relu(x)
    print int(x.shape[-1]) * int(x.shape[-2])
    x = Reshape((16,9*128))(x)
    
    x = Bidirectional(CuDNNLSTM(128,return_sequences=False))(x)



    
    return x

In [ ]:
# p = 0.3

timesteps, input_dim , latent_dim = 32,256, 128

x_logml = Input(shape=(timesteps, input_dim)) #1 channel, 99 time, 161 freqs # S : np.ndarray [shape=(n_mels, t)]

x = get_conv_stacks(x_logml)
# speaker_embedding = embedding(x_logml)


# x = concatenate([x,speaker_embedding])
x = Dense(128, activation = 'relu')(x) #
x = Dropout(0.3)(x)



x = Dense(len(POSSIBLE_LABELS), activation = 'softmax', name='targets')(x)




In [ ]:

model = Model(inputs = x_logml, outputs = x)
model.compile(optimizer=Adam(lr=5*1e-5), loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
model.load_weights("weights/conv2d_wlstm_hardpseudo_withval.hdf5")

In [ ]:
from keras.callbacks import TensorBoard
exp_name = "conv2d_wlstm_hardpseudo" #max_freqconvs_2510_avgshortcuts
callbacks = [EarlyStopping(monitor='val_loss',
                           patience=5,
                           verbose=1),
             
             ReduceLROnPlateau(monitor='val_loss',
                               factor=0.1,
                               patience=3,
                               verbose=1,
                               epsilon=0.01,
                              min_lr=1e-5),
             ModelCheckpoint(monitor='val_loss',
                             filepath='weights/{}.hdf5'.format(exp_name),
                             save_best_only=True,
                             save_weights_only=True,
                             mode='min')
#              , TensorBoard(log_dir='./logs/logs_{}'.format(exp_name), histogram_freq=0, batch_size=64, write_graph=True)
            ]

soft pseudo:
Epoch 4/100
329/329 [==============================] - 105s 320ms/step - loss: 0.1513 - acc: 0.9739 - val_loss: 0.0916 - val_acc: 0.9724

soft pseudo - with val Epoch 10/10
329/329 [==============================] - 105s 320ms/step - loss: 0.1532 - acc: 0.9735 - val_loss: 0.0827 - val_acc: 0.9753


hard pseudo 
Epoch 4/100
329/329 [==============================] - 105s 321ms/step - loss: 0.0783 - acc: 0.9731 - val_loss: 0.0953 - val_acc: 0.9743


hard pseudo + val 
Epoch 10/100
329/329 [==============================] - 106s 323ms/step - loss: 0.0637 - acc: 0.9801 - val_loss: 0.0532 - val_acc: 0.9850

In [ ]:
batch_size = 64

history = model.fit_generator(generator=mi,#train_generator(batch_size),
                              steps_per_epoch=train_df.shape[0]*(1./5)//batch_size,
                              epochs=100,
                              callbacks=callbacks,
                              validation_data=valid_generator(batch_size),
                              validation_steps=int(np.ceil(valid_df.shape[0]/batch_size)))

In [ ]:
model.load_weights('./weights/{}.hdf5'.format(exp_name))

In [ ]:
mi = MixIterator([train_generator(45), valid_generator(5),test_pseudo_generator(14)])


In [ ]:
from keras.callbacks import TensorBoard
exp_name = "conv2d_wlstm_hardpseudo_withval_c2" #max_freqconvs_2510_avgshortcuts
callbacks = [EarlyStopping(monitor='val_loss',
                           patience=5,
                           verbose=1),
             
             ReduceLROnPlateau(monitor='val_loss',
                               factor=0.1,
                               patience=3,
                               verbose=1,
                               epsilon=0.01,
                              min_lr=1e-5),
             ModelCheckpoint(monitor='val_loss',
                             filepath='weights/{}.hdf5'.format(exp_name),
                             save_best_only=True,
                             save_weights_only=True,
                             mode='min')
#              , TensorBoard(log_dir='./logs/logs_{}'.format(exp_name), histogram_freq=0, batch_size=64, write_graph=True)
            ]

In [ ]:
batch_size = 64

history = model.fit_generator(generator=mi,#train_generator(batch_size),
                              steps_per_epoch=train_df.shape[0]*(1./5)//batch_size,
                              epochs=100,
                              callbacks=callbacks,
                              validation_data=valid_generator(batch_size),
                              validation_steps=int(np.ceil(valid_df.shape[0]/batch_size)))

In [ ]:
model.load_weights('./weights/conv2d_wlstm_hardpseudo_withval_c2.hdf5')

### valid evaluation

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report

In [ ]:
val_preds = model.predict_generator(valid_generator(64),steps=int(np.ceil(valid_df.shape[0]/64.)))
val_preds = np.argmax(val_preds,axis=1)

In [ ]:
val_preds.shape

In [ ]:
oe = OneHotEncoder()

In [ ]:
y_true = oe.fit_transform(valid_df.label_id.values.reshape(-1, 1)).todense()



In [ ]:
val_preds = oe.transform(val_preds.reshape(-1, 1)).todense()


In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

#### tta test

In [ ]:


def augment_wav(wav,pval=0.5):
    sample_rate = 16000
    L = 1000 #16000  # 1 sec
    
#     adjust speed, with 50% chance
    wav = speed_change(wav,1.+ random.choice([.1,-0.1,0])) #random.uniform(-1, 1)*0.05) if np.random.random() < pval else wav
    
    
    #adjust volume
#     db_adjustment = random.uniform(-1, 1)*10
    wav = wav + random.choice([-10,-5,0,5,10]) #randodb_adjustment if np.random.random() < pval else wav
     
        
    #fill to 1 second
    wav = fill_to_1sec(wav)        
        
    #shift the audio by 10 ms
    shift_length = 100
    if np.random.random() < 0.5: #shift to left
        wav = wav[:L-shift_length]+ AudioSegment.silent(shift_length,frame_rate=sample_rate)
    else: #shift to right
        wav = AudioSegment.silent(shift_length,frame_rate=sample_rate) + wav[shift_length:]
        
        
        
    #blend original file with background noise     
#     if np.random.random() < pval:
    noise = random.choice(silence_files_AS)
    db_delta = (wav.dBFS - noise.dBFS) -10.

    if db_delta< 0: #reduce intensity of loud background; if it's too silent, leave it be
        noise = noise  + db_delta
    wav = wav.overlay(noise)
 
    return wav


def process_wav_file(record, reshape=False, augment=False,pval=0.5 ,output_format='logmel',n_mels=128 ):
    
    if type(record) == str: # test files
        fname = record
        label = "test"
    else:    
        fname  = record.wav_file
        label = record.label

        
        
        
        
    if "raw_AS_wav" in record: 
        wav = record.raw_AS_wav
    else:
        wav = AudioSegment.from_wav(fname.replace("\\","/"))
        
        
    
    if (not label in ["silence"]) and augment: #no augmentation for sample files 
        wav = augment_wav(wav,pval)

    else: #make sure segment is 1 second
        wav = fill_to_1sec(wav)

        
    samples = AS_to_raw(wav)
    
    
    
    if output_format == "logmel":
        output = log_mel(samples,reshape=reshape,n_mels=n_mels)
        
    elif output_format == "mfcc":
        log_S = log_mel(samples,reshape=False,n_mels=n_mels)
        mfcc = librosa.feature.mfcc(S=log_S, n_mfcc=40) #hirese mfcc
        delta1 = librosa.feature.delta(mfcc, order=1)#hirese mfcc
        delta2 = librosa.feature.delta(mfcc, order=2)

        output = np.stack([mfcc,delta1,delta2])
        
    elif  output_format == "cqt":   
        output = librosa.cqt(samples, sr=16000)
    else:
        output = samples
    
    
    return output


In [ ]:
silence_files_AS = [AudioSegment.from_wav(x) for x in silent_df.wav_file.values]

filler = AudioSegment.silent(duration=1000, frame_rate = 16000)

def valid_generator_aug(val_batch_size):
    while True:
        ids = list(range(valid_df.shape[0]))
        for start in range(0, len(ids), val_batch_size):
            x_batch = []
            y_batch = []
            end = min(start + val_batch_size, len(ids))
            i_val_batch = ids[start:end]
            for i in i_val_batch:
                x_batch.append(process_wav_file(valid_df.loc[i],n_mels=256,augment=True).T)
                y_batch.append(valid_df.label_id.values[i])

            x_batch = 1.- np.array(x_batch)/-80.
            y_batch = to_categorical(y_batch, num_classes = len(POSSIBLE_LABELS))
            yield x_batch, y_batch

In [ ]:
augs= []
num_augs = 2
for i in range(num_augs):
    tmp_gen = valid_generator_aug(64)
    augs.append(model.predict_generator(tmp_gen,steps=int(np.ceil(valid_df.shape[0]/64.)),verbose=1))
    del tmp_gen

In [ ]:
augs.append(val_preds)

In [ ]:
val_blend = np.mean(augs,axis=0) #np.mean([val_preds,preds_aug1],axis=0)
val_blend = np.argmax(val_blend,axis=1)
val_blend = oe.transform(val_blend.reshape(-1, 1)).todense()

In [ ]:
print classification_report(y_true,val_blend,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

In [ ]:
print classification_report(y_true,val_preds,target_names=POSSIBLE_LABELS)

### submission


In [ ]:
test_paths = glob(os.path.join('./data/', 'test/audio/*wav'))

In [ ]:
len(test_paths)

In [ ]:
test_df = pickle.load( open("cache/test_df_256.pik","rb"))


In [ ]:
test_df.shape

In [ ]:
test_df.head()

In [ ]:
test_paths

In [ ]:
batch_size = 64 

def test_generator(test_batch_size,augment=False):
    while True:
        ids = list(range(test_df.shape[0]))
        
        for start in range(0, len(ids), test_batch_size):
            x_batch = []
            end = min(start + test_batch_size, len(ids))
            i_test_batch = ids[start:end]
#             this_paths = test_paths[start:end]
#             for x in this_paths:
            for i in i_test_batch:
                if augment:
                    x_batch.append(process_wav_file(test_df.loc[i],n_mels=256,augment=True).T) #,reshape=False,augment=augment,pval=0.5))
                else:    
                    x_batch.append(test_df.loc[i,'raw'].T)

            x_batch = np.array(x_batch)
            x_batch = 1.- np.array(x_batch)/-80.
            
            yield x_batch

In [ ]:
predictions = model.predict_generator(test_generator(batch_size), int(np.ceil(len(test_paths)/float(batch_size))), verbose=1)

In [ ]:
np.save("cache/predictions_{}.npy".format(exp_name),predictions)

In [ ]:
predictions = np.load("cache/predictions_conv2d_wlstm_pseudo_wval.npy")

In [ ]:
tta_preds = []
num_aug = 4
for i in range(num_aug):
    tta_preds.append(model.predict_generator(test_generator(batch_size,augment=True), int(np.ceil(len(test_paths)/float(batch_size))), verbose=1))

In [ ]:
# import pickle

# pickle.dump(tta_preds,open("cache/test_preds_aug_4x.pik",'wb'),pickle.HIGHEST_PROTOCOL)

# tta_preds = pickle.load(open("cache/test_preds_aug_4x.pik",'rb'))

# tta_preds.append(predictions)

In [ ]:
predictions = np.mean(tta_preds,axis=0)

In [ ]:
predictions.shape

#### blending

In [ ]:
# predictions_pyramid_noaug = np.load('cache/predictions_pyramid_noaug.npy')
# predictions_model_with_ae_base_drp2_1 = np.load('cache/predictions_model_with_ae_base_drp2_1.npy')
# blend1 = np.load('cache/predictions_blend_dilated_conv1d_timek123_n_freqk48_pseudo_mixtimefreq1ds_plus_samewithlstm_plus_aebased_conv2d_finetuned.npy')
# predictions_aebase_aug_drp3_finetune = np.load('cache/predictions_aebase_aug_drp3_finetune.npy')
# predictions_conv_n_lstm = np.load("cache/predictions_dilated_conv_n_lstm.npy")
blend2 = np.load('cache/predictions_frqmaxpool256_plus_longblend1.npy')

In [ ]:
blend3 = np.mean([predictions,
                       blend2], axis=0)


In [ ]:
np.save("cache/predictions_{}_{}.npy".format(exp_name,"plus_blend2"),blend3)

In [ ]:
# num_aug = 2 
# for i in range(num_aug):
#     predictions +=  model.predict_generator(test_generator(64,augment=True), int(np.ceil(len(test_paths)/64.)), verbose=1)
    

# predictions = predictions/(num_aug + 1.0)

#### class assignment

In [ ]:
classes = np.argmax(blend3, axis=1)

In [ ]:
plt.hist(classes,bins=10)

In [ ]:
plt.hist(classes,bins=10)

In [ ]:
plt.hist(classes,bins=10)

In [ ]:
plt.hist(classes,bins=10)

### submission

In [ ]:
### last batch will contain padding, so remove duplicates
submission = dict()
for i in range(len(test_paths)):
    fname, label = os.path.basename(test_paths[i]), id2name[classes[i]]
    submission[fname] = label

In [ ]:
with open('subm/{}_plus_blend2.csv'.format(exp_name), 'w') as fout: #_blend_conv1dlstm_and_aebased_conv2d_finetuned
    fout.write('fname,label\n')
    for fname, label in submission.items():
        fout.write('{},{}\n'.format(fname, label))

In [ ]:
from IPython.lib.display import FileLink

FileLink('subm/{}_plus_blend2.csv'.format(exp_name))


#### creae logits 


In [ ]:
test_preds = np.load("cache/predictions_frqmaxpool256_plus_longblend1.npy")
# ("cache/predictions_conv2dlstm_plus_longblend1.npy") > best of 0.88

In [ ]:
classes = np.argmax(test_preds, axis=1)

In [ ]:
class_labels = []

for i in range(len(test_paths)):
    class_labels.append(id2name[classes[i]])
    


In [ ]:
test_file_names = []
for i in range(len(test_paths)):
    test_file_names.append(os.path.basename(test_paths[i]))


In [ ]:
test_file_names[0]

In [ ]:
test_preds[0]

In [ ]:
logits = pd.concat([pd.DataFrame(data=test_file_names,columns=["fname"]) ,
                    pd.DataFrame(data=class_labels,columns=["label"]), 
           pd.DataFrame(data = test_preds, columns=POSSIBLE_LABELS)],axis=1)

In [ ]:
logits.tail()

In [ ]:
logits.to_csv("cache/armin_logits_88.csv",index=False)